# LeetCode 948 — Bag of Tokens bằng Python

Link đề: https://leetcode.com/problems/bag-of-tokens/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: **mua điểm bằng token rẻ nhất, bán điểm lấy năng lượng bằng token đắt nhất** — sắp xếp rồi dùng hai con trỏ `lo` (rẻ) và `hi` (đắt); luôn **ghi nhận điểm cao nhất** từng đạt vì điểm cuối có thể thấp hơn đỉnh. Oracle: **tìm kiếm vét cạn mọi cách chơi** (bất kỳ token nào, ngửa hay úp) có nhớ trạng thái.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: khởi đầu `power` năng lượng, `score = 0`; mỗi token chơi **tối đa một lần**, theo một trong hai cách: **ngửa** (face-up: trả `tokens[i]` năng lượng, `+1` điểm, cần `power ≥ tokens[i]`) hoặc **úp** (face-down: `+tokens[i]` năng lượng, `−1` điểm, cần `score ≥ 1`); tối đa hoá điểm đạt được.
2. Nắm công cụ Python: `sorted`, hai con trỏ, `deque`, `heapq` (hai heap), quy hoạch động trên chỉ số `(i, j)`, tìm kiếm có nhớ để làm oracle.
3. Đi từ **sắp xếp + hai con trỏ tham lam** → **`deque` (`popleft`/`pop`)** → **hai heap** → **DP trên cặp `(i, j)` (đã chơi `i` token rẻ ngửa, `j` token đắt úp)**.
4. Hiểu mấu chốt: để lấy điểm ta nên tốn **ít** năng lượng nhất (token rẻ nhất), và để lấy năng lượng ta nên đổi điểm lấy **nhiều** năng lượng nhất (token đắt nhất); trạng thái hoàn toàn xác định bởi `(i, j)`.
5. Tránh lỗi kinh điển: trả điểm **cuối** thay vì **cực đại**, điều kiện vòng lặp `i < j` thay vì `i <= j` (bỏ sót token cuối cùng), bán token **rẻ** thay vì **đắt**, quên sắp xếp, bán điểm khi vẫn còn mua được, và quên trường hợp `tokens` rỗng.

## 2. Đề bài & ví dụ

Bạn có `power` năng lượng ban đầu, điểm `score = 0`, và một túi `tokens`. Mỗi token `tokens[i]` được chơi **tối đa một lần** theo **một** trong hai cách:

- **Ngửa (face-up):** nếu `power ≥ tokens[i]`: `power -= tokens[i]`, `score += 1`.
- **Úp (face-down):** nếu `score ≥ 1`: `power += tokens[i]`, `score -= 1`.

Trả về **điểm lớn nhất** có thể đạt được sau khi chơi một số token tuỳ ý (kể cả không chơi).

**Ví dụ 1:** `tokens = [100]`, `power = 50` → `0` (không đủ năng lượng; điểm `0` nên không thể úp).

**Ví dụ 2:** `tokens = [200,100]`, `power = 150` → `1` (ngửa token `100`).

**Ví dụ 3:** `tokens = [100,200,300,400]`, `power = 200` → `2`: ngửa `100` (power `100`, score `1`); úp `400` (power `500`, score `0`); ngửa `200` (power `300`, score `1`); ngửa `300` (power `0`, score `2`).

**Ràng buộc:** `0 <= tokens.length <= 1000`; `0 <= tokens[i], power < 10^4`.

In [1]:
import itertools
import random
import time
from collections import deque
from functools import lru_cache
from heapq import heapify, heappop, heappush
from typing import List

examples = [(([100], 50), 0), (([200, 100], 150), 1), (([100, 200, 300, 400], 200), 2)]
for (t, p), want in examples:
    print(f"tokens = {t}, power = {p} -> {want}")

tokens = [100], power = 50 -> 0
tokens = [200, 100], power = 150 -> 1
tokens = [100, 200, 300, 400], power = 200 -> 2


## 3. Kiến thức nền

### 3.1 Hai thao tác và cân bằng điểm–năng lượng

Ngửa đổi **năng lượng lấy điểm** (`−t` năng lượng, `+1` điểm); úp đổi **điểm lấy năng lượng** (`−1` điểm, `+t` năng lượng). Mỗi token chỉ chơi một lần nên mỗi giá trị `t` chỉ đóng góp **một** lần, theo chiều mua hoặc chiều bán. Điểm cuối cùng `= (số lần ngửa) − (số lần úp)`.

In [2]:
power, score = 200, 0
for tok, how in [(100, "up"), (400, "down"), (200, "up"), (300, "up")]:
    if how == "up":
        assert power >= tok
        power, score = power - tok, score + 1
    else:
        assert score >= 1
        power, score = power + tok, score - 1
    print(f"token {tok} ({'ngửa' if how == 'up' else 'úp'}): power = {power}, score = {score}")

token 100 (ngửa): power = 100, score = 1
token 400 (úp): power = 500, score = 0
token 200 (ngửa): power = 300, score = 1
token 300 (ngửa): power = 0, score = 2


### 3.2 Sắp xếp và hai con trỏ

Sau khi sắp xếp tăng, `tokens[lo]` là token **rẻ nhất** còn lại, `tokens[hi]` là **đắt nhất**. Mua điểm: dùng `tokens[lo]` (tốn ít nhất); bán điểm: dùng `tokens[hi]` (thu nhiều nhất).

### 3.3 Cực đại, không phải giá trị cuối

Sau khi úp (bán điểm) điểm giảm nhưng có thể mở ra cơ hội mua nhiều hơn; kết thúc có thể ở điểm thấp hơn đỉnh đã đạt. Vì vậy cần `best = max(best, score)` ở **mọi** thời điểm.

### 3.4 Trạng thái `(i, j)`

Khi chỉ chơi token rẻ nhất ngửa và token đắt nhất úp, sau khi đã ngửa `i` token đầu (`sorted[:i]`) và úp `j` token cuối (`sorted[n−j:]`): `power = P − prefix[i] + suffix[j]`, `score = i − j` — hoàn toàn xác định.

In [3]:
tokens, P = sorted([100, 200, 300, 400]), 200
prefix = [0] + list(itertools.accumulate(tokens))
suffix = [0] + list(itertools.accumulate(reversed(tokens)))
for i, j in [(0, 0), (1, 0), (1, 1), (2, 1), (3, 1)]:
    print(f"(i={i}, j={j}): power = {P} - {prefix[i]} + {suffix[j]} = {P - prefix[i] + suffix[j]}, score = {i - j}")

(i=0, j=0): power = 200 - 0 + 0 = 200, score = 0
(i=1, j=0): power = 200 - 100 + 0 = 100, score = 1
(i=1, j=1): power = 200 - 100 + 400 = 500, score = 0
(i=2, j=1): power = 200 - 300 + 400 = 300, score = 1
(i=3, j=1): power = 200 - 600 + 400 = 0, score = 2


## 4. Ý tưởng nền tảng / trực giác

`tokens = [100, 200, 300, 400]`, `power = 200`. Sắp xếp rồi theo dõi:

| bước | hành động | `power` | `score` | `best` |
|---|---|---|---|---|
| 1 | ngửa token rẻ nhất `100` (đủ năng lượng) | 100 | 1 | 1 |
| 2 | `200` đắt hơn `power`: úp token đắt nhất `400` | 500 | 0 | 1 |
| 3 | ngửa `200` | 300 | 1 | 1 |
| 4 | ngửa `300` | 0 | 2 | **2** |

Đáp án `2`. Quan sát: bán `400` (đắt nhất) lấy nhiều năng lượng nhất để mua được hai token `200`, `300`.

In [4]:
tk = sorted([100, 200, 300, 400])
lo, hi, power, score, best = 0, len(tk) - 1, 200, 0, 0
while lo <= hi:
    if power >= tk[lo]:
        power -= tk[lo]; score += 1; lo += 1
        action = f"ngửa {tk[lo - 1]}"
    elif score > 0:
        power += tk[hi]; score -= 1; hi -= 1
        action = f"úp {tk[hi + 1]}"
    else:
        break
    best = max(best, score)
    print(f"{action:10s}: power = {power}, score = {score}, best = {best}")

ngửa 100  : power = 100, score = 1, best = 1
úp 400    : power = 500, score = 0, best = 1
ngửa 200  : power = 300, score = 1, best = 1
ngửa 300  : power = 0, score = 2, best = 2


## 5. Cách 1 — Sắp xếp + hai con trỏ

**Ý tưởng:** sắp xếp `tokens`. `lo = 0`, `hi = n − 1`, `score = best = 0`. Trong khi `lo ≤ hi`: nếu `power ≥ tokens[lo]` thì **ngửa** token `lo`; ngược lại nếu `score > 0` thì **úp** token `hi`; ngược lại dừng. Cập nhật `best` sau mỗi bước.

In [5]:
def bag_two_pointers(tokens, power):
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo <= hi:                                      # lo <= hi: token cuối cùng (lo == hi) vẫn có thể chơi
        if power >= tokens[lo]:
            power -= tokens[lo]                          # ngửa token rẻ nhất: mua 1 điểm
            score += 1
            lo += 1
            best = max(best, score)                      # đỉnh điểm chỉ tăng ở bước ngửa
        elif score > 0:
            power += tokens[hi]                          # úp token đắt nhất: đổi 1 điểm lấy nhiều năng lượng nhất
            score -= 1
            hi -= 1
        else:
            break                                        # không mua được, không bán được
    return best


for (t, p), want in examples:
    assert bag_two_pointers(t, p) == want
print([bag_two_pointers(t, p) for (t, p), _ in examples])

[0, 1, 2]


In [6]:
# Code nộp LeetCode cho Cách 1
class SolutionTwoPointers:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        tokens.sort()
        lo, hi = 0, len(tokens) - 1
        score = best = 0
        while lo <= hi:
            if power >= tokens[lo]:
                power -= tokens[lo]
                score += 1
                lo += 1
                best = max(best, score)
            elif score > 0:
                power += tokens[hi]
                score -= 1
                hi -= 1
            else:
                break
        return best

**Độ phức tạp:** `O(n log n)` (sắp xếp) + `O(n)` (hai con trỏ); `O(1)` bộ nhớ thêm.

**Nút thắt:** cần chứng minh tham lam (mục 10); và dạng khác cho góc nhìn khác (hàng đợi hai đầu, heap, DP).

## 6. Ý tưởng thuật toán / bước đột phá

Mua điểm bằng **token rẻ nhất**, bán điểm cho **token đắt nhất**; chỉ bán khi **không mua được**. Ghi nhận cực đại điểm. Mỗi token dùng một lần: hai con trỏ gặp nhau là hết.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** `deque`, **Cách 3** hai heap, **Cách 4** DP trên cặp `(i, j)`.

### Cách 2 — `deque` (`popleft` / `pop`)

**Ý tưởng:** cùng quy tắc, nhưng lưu token đã sắp xếp trong `deque`: `popleft()` lấy rẻ nhất, `pop()` lấy đắt nhất. Điều kiện dừng: `deque` rỗng.

In [7]:
def bag_deque(tokens, power):
    dq = deque(sorted(tokens))
    score = best = 0
    while dq:
        if power >= dq[0]:
            power -= dq.popleft()                        # ngửa token rẻ nhất
            score += 1
            best = max(best, score)
        elif score > 0:
            power += dq.pop()                            # úp token đắt nhất
            score -= 1
        else:
            break
    return best


for (t, p), want in examples:
    assert bag_deque(t, p) == want
print([bag_deque(t, p) for (t, p), _ in examples])

[0, 1, 2]


In [8]:
# Code nộp LeetCode cho Cách 2
from collections import deque


class SolutionDeque:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        dq = deque(sorted(tokens))
        score = best = 0
        while dq:
            if power >= dq[0]:
                power -= dq.popleft()
                score += 1
                best = max(best, score)
            elif score > 0:
                power += dq.pop()
                score -= 1
            else:
                break
        return best

**Độ phức tạp:** `O(n log n)`; `O(n)` bộ nhớ cho `deque`.

### Cách 3 — Hai heap (min-heap và max-heap)

**Ý tưởng:** không sắp xếp toàn bộ; dùng **min-heap** để lấy token rẻ nhất và **max-heap** (khoá âm) để lấy token đắt nhất. Mỗi token ở **cả hai** heap; ta đánh dấu đã dùng bằng bộ đếm để bỏ qua bản sao lỗi thời (xoá lười).

In [9]:
def bag_heaps(tokens, power):
    minh = list(enumerate(tokens))
    minh = [(v, i) for i, v in minh]
    maxh = [(-v, i) for v, i in minh]
    heapify(minh)
    heapify(maxh)
    used = [False] * len(tokens)
    score = best = 0

    def peek(h, sign):
        while h and used[h[0][1]]:                       # bỏ các phần tử đã dùng (xoá lười)
            heappop(h)
        return sign * h[0][0] if h else None

    while True:
        cheap, dear = peek(minh, 1), peek(maxh, -1)
        if cheap is None:
            break
        if power >= cheap:
            v, i = heappop(minh)
            used[i] = True
            power -= v
            score += 1
            best = max(best, score)
        elif score > 0:
            nv, i = heappop(maxh)
            used[i] = True
            power += -nv
            score -= 1
        else:
            break
    return best


for (t, p), want in examples:
    assert bag_heaps(t, p) == want
print([bag_heaps(t, p) for (t, p), _ in examples])

[0, 1, 2]


In [10]:
# Code nộp LeetCode cho Cách 3
from heapq import heapify, heappop


class SolutionHeaps:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        minh = [(v, i) for i, v in enumerate(tokens)]
        maxh = [(-v, i) for v, i in minh]
        heapify(minh)
        heapify(maxh)
        used = [False] * len(tokens)
        score = best = 0
        while True:
            while minh and used[minh[0][1]]:
                heappop(minh)
            while maxh and used[maxh[0][1]]:
                heappop(maxh)
            if not minh:
                break
            if power >= minh[0][0]:
                v, i = heappop(minh)
                used[i] = True
                power -= v
                score += 1
                best = max(best, score)
            elif score > 0:
                nv, i = heappop(maxh)
                used[i] = True
                power -= nv
                score -= 1
            else:
                break
        return best

**Độ phức tạp:** `O(n log n)` (mỗi token vào/ra mỗi heap tối đa một lần); `O(n)` bộ nhớ.

### Cách 4 — DP trên cặp `(i, j)`

**Ý tưởng:** theo mục 3.4, trạng thái `(i, j)` (đã ngửa `i` token rẻ nhất, úp `j` token đắt nhất, `i + j ≤ n`) xác định `power` và `score`. Từ `(i, j)`: **ngửa** token `i` nếu `power ≥ tokens[i]` ⇒ `(i + 1, j)`; **úp** token `n − 1 − j` nếu `score ≥ 1` và còn token (`i + j < n`) ⇒ `(i, j + 1)`. Đáp án = điểm lớn nhất `i − j` trên các trạng thái **tới được**. Cách này **không dùng** quy tắc "chỉ úp khi không mua được" — nó thử cả hai.

In [11]:
def bag_dp(tokens, power):
    tk = sorted(tokens)
    n = len(tk)
    prefix = [0] + list(itertools.accumulate(tk))
    suffix = [0] + list(itertools.accumulate(reversed(tk)))
    best = 0
    reach = {(0, 0)}
    frontier = [(0, 0)]
    while frontier:
        nxt = []
        for i, j in frontier:
            pw, sc = power - prefix[i] + suffix[j], i - j
            best = max(best, sc)
            if i + j < n:
                if pw >= tk[i] and (i + 1, j) not in reach:           # ngửa token rẻ nhất còn lại
                    reach.add((i + 1, j))
                    nxt.append((i + 1, j))
                if sc >= 1 and (i, j + 1) not in reach:               # úp token đắt nhất còn lại
                    reach.add((i, j + 1))
                    nxt.append((i, j + 1))
        frontier = nxt
    return best


for (t, p), want in examples:
    assert bag_dp(t, p) == want
print([bag_dp(t, p) for (t, p), _ in examples])

[0, 1, 2]


In [12]:
# Code nộp LeetCode cho Cách 4 (O(n^2) trạng thái: thử cả hai lựa chọn mỗi bước)
class SolutionDP:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        tk = sorted(tokens)
        n = len(tk)
        prefix = [0]
        for v in tk:
            prefix.append(prefix[-1] + v)
        suffix = [0]
        for v in reversed(tk):
            suffix.append(suffix[-1] + v)
        best, reach, frontier = 0, {(0, 0)}, [(0, 0)]
        while frontier:
            nxt = []
            for i, j in frontier:
                pw, sc = power - prefix[i] + suffix[j], i - j
                best = max(best, sc)
                if i + j < n:
                    if pw >= tk[i] and (i + 1, j) not in reach:
                        reach.add((i + 1, j))
                        nxt.append((i + 1, j))
                    if sc >= 1 and (i, j + 1) not in reach:
                        reach.add((i, j + 1))
                        nxt.append((i, j + 1))
            frontier = nxt
        return best

**Độ phức tạp:** tối đa `O(n²)` trạng thái `(i, j)`, `O(1)` mỗi trạng thái; `O(n²)` bộ nhớ trong trường hợp xấu. Dùng như kiểm chứng độc lập cho tham lam (cho `n ≤ ~1000`, vẫn chạy được).

## 8. TRACE từng bước

**Trace Cách 1** với `tokens = [100,200,300,400]`, `power = 200`: đã in ở mục 4. **Trace Cách 4** (DP `(i, j)`): từ `(0,0)`: `power 200 ≥ 100` ⇒ ngửa `(1,0)` (`power 100`, `score 1`). Từ `(1,0)`: `100 ≥ 200`? không; `score 1 ≥ 1` ⇒ úp `(1,1)` (`power 500`, `score 0`). Từ `(1,1)`: `500 ≥ 200` ⇒ `(2,1)` (`power 300`, `score 1`); `score 0` nên không úp. Từ `(2,1)`: `300 ≥ 300` ⇒ `(3,1)` (`power 0`, `score 2`) ⇒ `best = 2`.

Hàm trace in từng trạng thái của DP:

In [13]:
def trace_dp(tokens, power):
    tk = sorted(tokens)
    n = len(tk)
    prefix = [0] + list(itertools.accumulate(tk))
    suffix = [0] + list(itertools.accumulate(reversed(tk)))
    best, reach, frontier = 0, {(0, 0)}, [(0, 0)]
    print(f"tokens đã sắp: {tk}, power = {power}")
    while frontier:
        nxt = []
        for i, j in frontier:
            pw, sc = power - prefix[i] + suffix[j], i - j
            best = max(best, sc)
            moves = []
            if i + j < n:
                if pw >= tk[i]:
                    moves.append(f"ngửa {tk[i]} -> ({i + 1},{j})")
                    if (i + 1, j) not in reach:
                        reach.add((i + 1, j)); nxt.append((i + 1, j))
                if sc >= 1:
                    moves.append(f"úp {tk[n - 1 - j]} -> ({i},{j + 1})")
                    if (i, j + 1) not in reach:
                        reach.add((i, j + 1)); nxt.append((i, j + 1))
            print(f"  ({i},{j}): power {pw}, score {sc}; {', '.join(moves) or 'hết nước đi'}")
        frontier = nxt
    print("=> điểm lớn nhất:", best)
    return best


assert trace_dp([100, 200, 300, 400], 200) == 2

tokens đã sắp: [100, 200, 300, 400], power = 200
  (0,0): power 200, score 0; ngửa 100 -> (1,0)
  (1,0): power 100, score 1; úp 400 -> (1,1)
  (1,1): power 500, score 0; ngửa 200 -> (2,1)
  (2,1): power 300, score 1; ngửa 300 -> (3,1), úp 300 -> (2,2)
  (3,1): power 0, score 2; hết nước đi
  (2,2): power 600, score 0; hết nước đi
=> điểm lớn nhất: 2


## 9. Code nộp LeetCode

Chọn **Cách 1** (sắp xếp + hai con trỏ). Giữ nguyên `class Solution`, `bagOfTokensScore(self, tokens, power)`:

In [14]:
class Solution:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        tokens.sort()
        lo, hi = 0, len(tokens) - 1
        score = best = 0
        while lo <= hi:                                    # lo == hi: còn đúng một token, vẫn có thể ngửa
            if power >= tokens[lo]:
                power -= tokens[lo]                        # mua điểm bằng token rẻ nhất
                score += 1
                lo += 1
                best = max(best, score)                    # điểm đỉnh
            elif score > 0:
                power += tokens[hi]                        # bán điểm lấy năng lượng của token đắt nhất
                score -= 1
                hi -= 1
            else:
                break
        return best


print([Solution().bagOfTokensScore(list(t), p) for (t, p), _ in examples])      # [0, 1, 2]
assert all(Solution().bagOfTokensScore(list(t), p) == want for (t, p), want in examples)

[0, 1, 2]


## 10. Vì sao đúng

**(a) Trạng thái đủ gọn.** Mọi chiến lược là một chuỗi hành động "ngửa token `a`" / "úp token `b`". Ta chứng minh có chiến lược tối ưu chỉ dùng **token rẻ nhất còn lại để ngửa** và **token đắt nhất còn lại để úp**.

**(b) Ngửa luôn dùng token rẻ nhất.** Giả sử một chiến lược ngửa token `a` trong khi có token `c < a` chưa chơi lúc đó (hoặc chơi sau). Đổi vai: ngửa `c` thay `a` và dành `a` cho vai trò mà `c` đảm nhiệm sau đó (nếu `c` được úp: úp `a` thay vì `c` cho **nhiều** năng lượng hơn — không xấu hơn; nếu `c` ngửa sau: hoán đổi thứ tự hai lần ngửa). Năng lượng sau mỗi bước không giảm và điều kiện ngửa (`power ≥` giá) dễ thoả hơn với `c < a`. Vậy chiến lược không tệ hơn.

**(c) Úp luôn dùng token đắt nhất.** Úp cộng `t` năng lượng; dùng token đắt nhất cho năng lượng lớn nhất, và các token rẻ hơn được giữ lại để ngửa (rẻ hơn thì dễ ngửa hơn). Đổi chỗ tương tự (b).

**(d) Chỉ úp khi cần.** Nếu `power ≥ tokens[lo]` thì ngửa ngay (tăng điểm, tốn ít năng lượng nhất); hoãn ngửa để úp trước chỉ làm mất cơ hội (úp giảm điểm, và ngửa sau vẫn thực hiện được nếu năng lượng đủ; nhưng trong lúc đó `best` không tăng). Úp chỉ cần khi **không** thể ngửa (và có điểm). Điểm cực đại được ghi nhận sau mỗi lần ngửa (điểm chỉ tăng ở bước ngửa).

**(e) Dừng.** Khi không ngửa được (`power < tokens[lo]`) và không có điểm để úp (`score = 0`) thì bế tắc; không thể tăng thêm.

**(f) Cách 2, 3.** Cùng quy tắc, khác cấu trúc dữ liệu; heap dùng xoá lười để mỗi token chỉ dùng một lần.

**(g) Cách 4.** DP thử **cả** hai lựa chọn tại mỗi trạng thái nên không dựa vào (d); nhưng vẫn dựa vào (b), (c) (cấu trúc `(i, j)`). Oracle ở mục 12 **không** dựa vào cả (b)–(d): thử mọi token, mọi cách.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **tìm kiếm vét cạn** (bất kỳ token, ngửa hay úp) trên mọi mảng nhỏ và mọi `power` nhỏ (mục 12D), và ngẫu nhiên (12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp + hai con trỏ | `O(n log n)` | `O(1)`–`O(n)` | **đáp án chính** |
| 2. `deque` | `O(n log n)` | `O(n)` | cùng quy tắc |
| 3. Hai heap | `O(n log n)` | `O(n)` | không sắp xếp toàn bộ |
| 4. DP `(i, j)` | `O(n²)` | `O(n²)` xấu | kiểm chứng tham lam |

Đo thực tế (`n = 1000`):

In [15]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(list(args[0]), args[1])
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(948)
toks = [rng.randint(0, 9999) for _ in range(1000)]
for name, fn in [("two_pointers", bag_two_pointers), ("deque", bag_deque), ("heaps", bag_heaps), ("dp", bag_dp)]:
    print(f"n = 1000 {name:13s} {bench(fn, (toks, 3000)) * 1e3:8.2f} ms")

n = 1000 two_pointers      0.21 ms
n = 1000 deque             0.19 ms
n = 1000 heaps             1.04 ms


n = 1000 dp               75.65 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ × mọi `power` nhỏ với oracle tìm kiếm, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(tokens, power):
    """Tìm kiếm vét cạn: mọi token chưa chơi, ngửa hoặc úp, có nhớ trạng thái (mask, power, score). Trả điểm lớn nhất từng đạt."""
    n = len(tokens)
    seen = set()
    best = [0]

    def dfs(mask, pw, sc):
        if (mask, pw, sc) in seen:
            return
        seen.add((mask, pw, sc))
        best[0] = max(best[0], sc)
        for i in range(n):
            if mask >> i & 1:
                continue
            if pw >= tokens[i]:                           # ngửa
                dfs(mask | 1 << i, pw - tokens[i], sc + 1)
            if sc >= 1:                                   # úp
                dfs(mask | 1 << i, pw + tokens[i], sc - 1)

    dfs(0, power, 0)
    return best[0]


IMPLS = {
    "two_pointers": bag_two_pointers,
    "deque": bag_deque,
    "heaps": bag_heaps,
    "dp": bag_dp,
    "Solution": lambda t, p: Solution().bagOfTokensScore(list(t), p),
    "SolutionTwoPointers": lambda t, p: SolutionTwoPointers().bagOfTokensScore(list(t), p),
    "SolutionDeque": lambda t, p: SolutionDeque().bagOfTokensScore(list(t), p),
    "SolutionHeaps": lambda t, p: SolutionHeaps().bagOfTokensScore(list(t), p),
    "SolutionDP": lambda t, p: SolutionDP().bagOfTokensScore(list(t), p),
}


def check(tokens, power, want=None, names=None):
    if want is None:
        want = oracle(tokens, power)
    for name in (names or IMPLS):
        got = IMPLS[name](list(tokens), power)
        assert got == want, (name, tokens, power, got, want)


for (t, p), want in examples:
    assert oracle(t, p) == want
    check(t, p, want)
print("ví dụ đề: oracle tìm kiếm vét cạn và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle tìm kiếm vét cạn và 9 cài đặt đúng


### B. Edge case

- `tokens` rỗng (ràng buộc cho phép độ dài `0`): `0`.
- Một token vừa đủ / không đủ năng lượng.
- `power = 0` và các token `0` (ngửa miễn phí tăng điểm).
- Tất cả token nhỏ hơn `power`: ngửa hết.
- Tất cả token lớn hơn `power`: `0`.
- Cần bán điểm để mua tiếp (ví dụ 3).
- Token cuối cùng (`lo == hi`) vẫn ngửa được.
- Điểm cuối thấp hơn điểm đỉnh.

In [17]:
cases = [
    ([], 10, 0), ([5], 5, 1), ([5], 4, 0), ([0], 0, 1), ([0, 0, 0], 0, 3), ([1, 2, 3], 100, 3),
    ([10, 20, 30], 5, 0), ([100, 200, 300, 400], 200, 2), ([1, 1, 1, 100], 3, 3), ([4, 4], 7, 1),
    ([2, 3, 100], 5, 2), ([50, 60, 70], 60, 1),
]
for t, p, want in cases:
    check(t, p, want)
# điểm cuối thấp hơn điểm đỉnh: [1,1,100] power 2: ngửa 1, 1 (điểm 2), rồi 100 không mua được; không cần bán
assert Solution().bagOfTokensScore([1, 1, 100], 2) == 2
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

12 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_final_score(tokens, power):                    # trả điểm CUỐI thay vì điểm cực đại
    tokens = sorted(tokens)
    lo, hi, score = 0, len(tokens) - 1, 0
    while lo <= hi:
        if power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1
        elif score > 0:
            power += tokens[hi]; score -= 1; hi -= 1
        else:
            break
    return score


def bug_lo_lt_hi(tokens, power):                       # while lo < hi: bỏ sót token cuối cùng
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo < hi:
        if power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1; best = max(best, score)
        elif score > 0:
            power += tokens[hi]; score -= 1; hi -= 1
        else:
            break
    return best


def bug_sell_cheapest(tokens, power):                  # úp token RẺ nhất thay vì đắt nhất
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo <= hi:
        if power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1; best = max(best, score)
        elif score > 0:
            power += tokens[lo]; score -= 1; lo += 1        # SAI: dùng token rẻ nhất để bán
        else:
            break
    return best


def bug_buy_dearest(tokens, power):                    # ngửa token ĐẮT nhất có thể (không phải rẻ nhất)
    tokens = sorted(tokens, reverse=True)
    score = best = 0
    pool = tokens[:]
    while pool:
        afford = [t for t in pool if t <= power]
        if afford:
            t = afford[0]                                   # đắt nhất mua được
            pool.remove(t)
            power -= t; score += 1; best = max(best, score)
        elif score > 0:
            power += pool.pop(0); score -= 1               # bán đắt nhất
        else:
            break
    return best


def bug_no_sort(tokens, power):                        # không sắp xếp
    lo, hi = 0, len(tokens) - 1
    tokens = list(tokens)
    score = best = 0
    while lo <= hi:
        if power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1; best = max(best, score)
        elif score > 0:
            power += tokens[hi]; score -= 1; hi -= 1
        else:
            break
    return best


def bug_sell_eagerly(tokens, power):                   # bán điểm ngay khi có thể, kể cả khi còn mua được
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo <= hi:
        if score > 0 and power < tokens[hi]:
            power += tokens[hi]; score -= 1; hi -= 1
        elif power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1; best = max(best, score)
        else:
            break
    return best


BUGS = [bug_final_score, bug_lo_lt_hi, bug_sell_cheapest, bug_buy_dearest, bug_no_sort, bug_sell_eagerly]
rng = random.Random(3)
tests = [([rng.randint(0, 6) for _ in range(rng.randint(0, 5))], rng.randint(0, 12)) for _ in range(2500)]
for bug in BUGS:
    bad = next(((t, p) for t, p in tests if bug(list(t), p) != oracle(t, p)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên tokens = {bad[0]}, power = {bad[1]}: ra {bug(list(bad[0]), bad[1])}, đúng {oracle(*bad)}")

bug_final_score    sai trên tokens = [4, 1], power = 3: ra 0, đúng 1
bug_lo_lt_hi       sai trên tokens = [4], power = 8: ra 0, đúng 1
bug_sell_cheapest  sai trên tokens = [2, 5, 3, 0, 4], power = 0: ra 1, đúng 2
bug_buy_dearest    sai trên tokens = [3, 4, 6, 4, 3], power = 6: ra 1, đúng 2
bug_no_sort        sai trên tokens = [4, 1], power = 3: ra 0, đúng 1
bug_sell_eagerly   sai trên tokens = [6, 1, 1, 5, 1], power = 8: ra 3, đúng 4


### D. Vét cạn mọi mảng nhỏ × mọi `power` nhỏ

Mọi `tokens` độ dài `0..5` với giá trị `0..3` (`1 + 4 + 16 + 64 + 256 + 1 024 = 1 365` mảng) × mọi `power` từ `0` tới `9` (`10` giá trị) = `13 650` bộ. Oracle: tìm kiếm có nhớ trạng thái.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("two_pointers", "deque", "heaps", "dp", "Solution")
for n in range(0, 6):
    for tup in itertools.product(range(4), repeat=n):
        for power in range(10):
            want = oracle(list(tup), power)
            for name in fast:
                assert IMPLS[name](list(tup), power) == want, (name, tup, power)
            cnt += 1
print(cnt, "bộ (tokens, power): 5 cài đặt khớp oracle tìm kiếm vét cạn;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 1365 * 10

13650 bộ (tokens, power): 5 cài đặt khớp oracle tìm kiếm vét cạn; 1.1 s


### E. Ngẫu nhiên có seed và tính chất

Dữ liệu ngẫu nhiên: `n ≤ 8` (so oracle) và `n` lớn (so các cách với nhau). Tính chất: (1) hoán vị `tokens` không đổi đáp án; (2) **tăng `power`** không làm đáp án giảm (đơn điệu); (3) **thêm token** không làm đáp án giảm; (4) đáp án `≤ n`; (5) nếu `power ≥ Σ tokens` thì đáp án `= n` (ngửa được mọi token).

In [20]:
rng = random.Random(20240948)
for _ in range(1500):
    n = rng.randint(0, 8)
    hi = rng.choice([3, 10, 30])
    tk = [rng.randint(0, hi) for _ in range(n)]
    power = rng.randint(0, 3 * hi)
    want = oracle(tk, power)
    for name in ("two_pointers", "deque", "heaps", "dp", "Solution", "SolutionTwoPointers", "SolutionDeque", "SolutionHeaps", "SolutionDP"):
        assert IMPLS[name](list(tk), power) == want, (name, tk, power)
    sh = tk[:]
    rng.shuffle(sh)
    assert Solution().bagOfTokensScore(sh, power) == want                                    # (1)
    assert Solution().bagOfTokensScore(list(tk), power + rng.randint(1, 10)) >= want         # (2)
    assert Solution().bagOfTokensScore(list(tk) + [rng.randint(0, hi)], power) >= want       # (3)
    assert want <= n                                                                          # (4)
    assert Solution().bagOfTokensScore(list(tk), sum(tk)) == n                                # (5)
for _ in range(100):
    n = rng.randint(50, 300)
    tk = [rng.randint(0, 9999) for _ in range(n)]
    power = rng.randint(0, 9999)
    ref = bag_two_pointers(tk, power)
    assert bag_deque(tk, power) == bag_heaps(tk, power) == bag_dp(tk, power) == Solution().bagOfTokensScore(list(tk), power) == ref
print("1500 bộ ngẫu nhiên khớp oracle; 5 tính chất (hoán vị, đơn điệu theo power/token, cận, đủ năng lượng) và 100 bộ tới 300 token đúng")

1500 bộ ngẫu nhiên khớp oracle; 5 tính chất (hoán vị, đơn điệu theo power/token, cận, đủ năng lượng) và 100 bộ tới 300 token đúng


### F. Giới hạn: `n = 1000`

Các cách khớp nhau trên nhiều dạng: token ngẫu nhiên, toàn `0`, toàn `9999`, tăng dần/giảm dần, `power` nhỏ/lớn; công thức đóng cho vài dạng.

In [21]:
rng = random.Random(500)
n = 1000
fast = ["two_pointers", "deque", "heaps", "dp", "Solution", "SolutionTwoPointers", "SolutionDeque", "SolutionHeaps", "SolutionDP"]
datasets = {
    "ngẫu nhiên": [rng.randint(0, 9999) for _ in range(n)],
    "toàn 0": [0] * n,
    "toàn 9999": [9999] * n,
    "tăng dần": list(range(n)),
    "giảm dần": list(range(n, 0, -1)),
    "nhỏ và lớn": [1] * 500 + [9999] * 500,
}
for name, tk in datasets.items():
    for power in (0, 1, 500, 9999):
        res = {k: IMPLS[k](tk, power) for k in fast}
        assert len(set(res.values())) == 1, (name, power, res)
    print(f"{name:12s}: power 500 -> {IMPLS['two_pointers'](tk, 500)}")
assert Solution().bagOfTokensScore([0] * n, 0) == n
assert Solution().bagOfTokensScore([9999] * n, 9998) == 0
assert Solution().bagOfTokensScore([1] * 500 + [9999] * 500, 500) == 500
print("công thức đóng khớp")

ngẫu nhiên  : power 500 -> 395


toàn 0      : power 500 -> 1000
toàn 9999   : power 500 -> 0


tăng dần    : power 500 -> 415


giảm dần    : power 500 -> 414


nhỏ và lớn  : power 500 -> 500
công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả điểm cuối thay vì điểm cực đại

In [22]:
print("[100,200,300,400], 200: điểm cuối =", bug_final_score([100, 200, 300, 400], 200), "| cực đại =", Solution().bagOfTokensScore([100, 200, 300, 400], 200))
bad = next((t, p) for t, p in tests if bug_final_score(list(t), p) != oracle(t, p))
print("phản ví dụ:", bad, "->", bug_final_score(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))

[100,200,300,400], 200: điểm cuối = 2 | cực đại = 2
phản ví dụ: ([4, 1], 3) -> 0 | đúng: 1


**Vì sao sai:** sau khi bán điểm, điểm cuối có thể thấp hơn điểm đã đạt (đề hỏi **điểm lớn nhất đạt được**). **Đúng:** `best = max(best, score)` sau mỗi lần ngửa.

### Lỗi 2: `while lo < hi` bỏ sót token cuối cùng

In [23]:
print("[5], 5:", bug_lo_lt_hi([5], 5), "| đúng:", Solution().bagOfTokensScore([5], 5))
assert bug_lo_lt_hi([5], 5) == 0 and Solution().bagOfTokensScore([5], 5) == 1

[5], 5: 0 | đúng: 1


**Vì sao sai:** khi `lo == hi` vẫn còn đúng một token chưa chơi. **Đúng:** `while lo <= hi`.

### Lỗi 3: bán token rẻ nhất

In [24]:
bad = next((t, p) for t, p in tests if bug_sell_cheapest(list(t), p) != oracle(t, p))
print(bad, "->", bug_sell_cheapest(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))

([2, 5, 3, 0, 4], 0) -> 1 | đúng: 2


**Vì sao sai:** bán token **đắt** nhất cho nhiều năng lượng nhất và giữ token rẻ để mua điểm. **Đúng:** `power += tokens[hi]`.

### Lỗi 4: mua bằng token đắt nhất có thể

In [25]:
bad = next((t, p) for t, p in tests if bug_buy_dearest(list(t), p) != oracle(t, p))
print(bad, "->", bug_buy_dearest(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))

([3, 4, 6, 4, 3], 6) -> 1 | đúng: 2


**Vì sao sai:** mua điểm cần tốn **ít** năng lượng nhất. **Đúng:** dùng `tokens[lo]` (rẻ nhất).

### Lỗi 5: quên sắp xếp

In [26]:
bad = next((t, p) for t, p in tests if bug_no_sort(list(t), p) != oracle(t, p))
print(bad, "->", bug_no_sort(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))

([4, 1], 3) -> 0 | đúng: 1


**Vì sao sai:** hai con trỏ chỉ có nghĩa "rẻ nhất/đắt nhất" trên mảng đã sắp xếp. **Đúng:** `tokens.sort()`.

### Lỗi 6 (bonus): bán điểm quá sớm

In [27]:
bad = next((t, p) for t, p in tests if bug_sell_eagerly(list(t), p) != oracle(t, p))
print(bad, "->", bug_sell_eagerly(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))

([6, 1, 1, 5, 1], 8) -> 3 | đúng: 4


**Vì sao sai:** nếu còn mua được thì ngửa ngay; bán điểm chỉ khi bị kẹt. **Đúng:** ưu tiên ngửa, rồi mới úp.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "hai thao tác đối nghịch trên các phần tử, mỗi phần tử dùng một lần, tối đa hoá một giá trị" với phần tử đã sắp xếp → **hai con trỏ**: lấy phần tử có lợi nhất cho mỗi thao tác từ hai đầu.

**Mẹo:**

1. Hỏi: thao tác nào cần phần tử nhỏ, thao tác nào cần phần tử lớn?
2. Ghi nhận **cực đại** ở mọi thời điểm (không phải giá trị cuối).
3. `while lo <= hi` khi mỗi phần tử còn dùng được.
4. Ưu tiên thao tác "có lợi" (ngửa) rồi mới thao tác "tốn kém" (úp).
5. Kiểm chứng tham lam bằng DP trạng thái hoặc tìm kiếm vét cạn.
6. `tokens` có thể rỗng.

**Họ bài cùng pattern:** Boats to Save People (881), Assign Cookies (455), Two Sum II (167), Maximize Greatness of an Array (2592), Maximum Number of Points From Grid Queries, Minimum Cost to Hire K Workers (857), Maximum Performance of a Team (1383).

## 15. Biến thể

1. **Giá cố định mỗi lần úp:** úp chỉ được `+t` nhưng tốn `c` điểm.
2. **Không được úp quá `k` lần.**
3. **Cần đạt đúng điểm `s`** (có thể không tối ưu): có đạt được không.
4. **Chơi theo thứ tự cho trước** (không được chọn).
5. **Trả về dãy hành động** (ngửa/úp, token nào) tối ưu.

Cài biến thể 5 (dãy hành động, kèm hàm kiểm tra) và biến thể 2 (giới hạn số lần úp) với oracle:

In [28]:
def bag_actions(tokens, power):
    """Biến thể 5: danh sách hành động của chiến lược tham lam đạt điểm lớn nhất (cắt tại thời điểm đạt đỉnh)."""
    tk = sorted(tokens)
    lo, hi = 0, len(tk) - 1
    score = best = 0
    actions, best_len = [], 0
    while lo <= hi:
        if power >= tk[lo]:
            actions.append(("up", tk[lo])); power -= tk[lo]; score += 1; lo += 1
            if score > best:
                best, best_len = score, len(actions)
        elif score > 0:
            actions.append(("down", tk[hi])); power += tk[hi]; score -= 1; hi -= 1
        else:
            break
    return actions[:best_len], best


def replay(tokens, power, actions):
    """Kiểm tra dãy hành động hợp lệ (mỗi token dùng tối đa một lần) và trả điểm cuối."""
    pool = list(tokens)
    score = 0
    for how, t in actions:
        assert t in pool
        pool.remove(t)
        if how == "up":
            assert power >= t
            power, score = power - t, score + 1
        else:
            assert score >= 1
            power, score = power + t, score - 1
    return score


def oracle_limited_down(tokens, power, k):
    n = len(tokens)
    seen, best = set(), [0]

    def dfs(mask, pw, sc, used_down):
        if (mask, pw, sc, used_down) in seen:
            return
        seen.add((mask, pw, sc, used_down))
        best[0] = max(best[0], sc)
        for i in range(n):
            if mask >> i & 1:
                continue
            if pw >= tokens[i]:
                dfs(mask | 1 << i, pw - tokens[i], sc + 1, used_down)
            if sc >= 1 and used_down < k:
                dfs(mask | 1 << i, pw + tokens[i], sc - 1, used_down + 1)

    dfs(0, power, 0, 0)
    return best[0]


def bag_limited_down(tokens, power, k):
    tk = sorted(tokens)
    lo, hi = 0, len(tk) - 1
    score = best = used = 0
    while lo <= hi:
        if power >= tk[lo]:
            power -= tk[lo]; score += 1; lo += 1; best = max(best, score)
        elif score > 0 and used < k:
            power += tk[hi]; score -= 1; hi -= 1; used += 1
        else:
            break
    return best


rng = random.Random(15)
for _ in range(400):
    tk = [rng.randint(0, 6) for _ in range(rng.randint(0, 6))]
    p = rng.randint(0, 12)
    acts, sc = bag_actions(tk, p)
    assert sc == oracle(tk, p) and replay(tk, p, acts) == sc
    for k in (0, 1, 2):
        assert bag_limited_down(tk, p, k) == oracle_limited_down(tk, p, k), (tk, p, k)
print("dãy hành động tối ưu hợp lệ; giới hạn số lần úp k = 0, 1, 2: tham lam khớp oracle (400 bộ)")

dãy hành động tối ưu hợp lệ; giới hạn số lần úp k = 0, 1, 2: tham lam khớp oracle (400 bộ)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp + hai con trỏ | ngửa rẻ nhất, úp đắt nhất | `O(n log n)` | `O(1)` | **đáp án chính** |
| 2. `deque` | `popleft` / `pop` | `O(n log n)` | `O(n)` | cùng quy tắc |
| 3. Hai heap | min-heap + max-heap, xoá lười | `O(n log n)` | `O(n)` | không sắp xếp toàn bộ |
| 4. DP `(i, j)` | thử cả ngửa lẫn úp | `O(n²)` | `O(n²)` | kiểm chứng tham lam |

Cả bốn khớp oracle vét cạn trên 13 650 bộ nhỏ và khớp nhau với `n = 1000` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().bagOfTokensScore([50, 60, 70], 100), Solution().bagOfTokensScore([10, 100, 200, 300], 150), Solution().bagOfTokensScore([], 5), Solution().bagOfTokensScore([0, 0], 0))

1 2 0 2


### Bài 2 (tính tay)

`tokens = [20, 50, 80, 130]`, `power = 60`. (a) Ghi từng bước hai con trỏ. (b) Điểm đỉnh? (c) Điểm cuối cùng? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai con trỏ:

In [30]:
def two_pointers_todo(tokens, power):
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo <= hi:
        pass            # TODO 1: nếu power >= tokens[lo]: ngửa (power, score, lo, best)
        pass            # TODO 2: elif score > 0: úp token hi
        pass            # TODO 3: else break
    return best


# Khi làm xong, bỏ comment để kiểm tra (nhớ thay `pass` bằng khối if/elif/else):
# assert [two_pointers_todo(list(t), p) for (t, p), _ in examples] == [0, 1, 2]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`deque`**:

In [31]:
class MySolution:
    def bagOfTokensScore(self, tokens, power):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().bagOfTokensScore(list(t), p) == w for (t, p), w in examples)

### Bài 5 (mini challenge): Maximize Greatness of an Array (2592)

Cho `nums`, hoán vị thành `perm` sao cho số chỉ số `i` có `perm[i] > nums[i]` là lớn nhất. Viết `maximize_greatness(nums)` bằng sắp xếp + hai con trỏ (cùng kiểu tham lam) và kiểm tra bằng vét cạn hoán vị.

In [32]:
def maximize_greatness(nums):
    pass            # TODO


# Khi làm xong:
# assert maximize_greatness([1, 3, 5, 2, 1, 3, 1]) == 4 and maximize_greatness([1, 2, 3, 4]) == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: sắp xếp rồi hai con trỏ.
- Mức 2: `[50,60,70]`, `100`: ngửa `50` (power 50, score 1); `60 > 50`, úp `70` (power 120, score 0); ngửa `60` (power 60, score 1) ⇒ `best = 1`.
- Mức 3: kết quả `1 2 0 2`: `[0,0]`, `0` ⇒ ngửa cả hai `0` ⇒ `2`; `[]` ⇒ `0`.

**Bài 2**

- Mức 1: `[20,50,80,130]`, `power = 60`.
- Mức 2: ngửa `20` (power 40, score 1); `50 > 40`: úp `130` (power 170, score 0); ngửa `50` (power 120, score 1); ngửa `80` (power 40, score 2).
- Mức 3: điểm đỉnh `2`, điểm cuối `2`.

**Bài 3**

- Mức 1: `if power >= tokens[lo]`.
- Mức 2: `elif score > 0`.
- Mức 3: `else: break`; cập nhật `best` khi ngửa.

**Bài 4**

- Mức 1: `dq = deque(sorted(tokens))`.
- Mức 2: so `dq[0]`.
- Mức 3: `popleft()` ngửa; `pop()` úp.

**Bài 5**

- Mức 1: sắp xếp hai bản `nums` và `perm`.
- Mức 2: con trỏ `j` trên `perm`: mỗi `perm[j]` lớn hơn `nums[i]` hiện tại thì `i += 1`.
- Mức 3: trả `i`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().bagOfTokensScore([50, 60, 70], 100), Solution().bagOfTokensScore([10, 100, 200, 300], 150), Solution().bagOfTokensScore([], 5), Solution().bagOfTokensScore([0, 0], 0))
assert vals == (1, 2, 0, 2) == (oracle([50, 60, 70], 100), oracle([10, 100, 200, 300], 150), oracle([], 5), oracle([0, 0], 0))
print("Bài 1 OK:", vals)

Bài 1 OK: (1, 2, 0, 2)


### Bài 2

`ngửa 20 → úp 130 → ngửa 50 → ngửa 80`; điểm đỉnh `2`, điểm cuối `2`.

In [34]:
tk, power = [20, 50, 80, 130], 60
lo, hi, score, best, log = 0, 3, 0, 0, []
while lo <= hi:
    if power >= tk[lo]:
        log.append(("up", tk[lo])); power -= tk[lo]; score += 1; lo += 1; best = max(best, score)
    elif score > 0:
        log.append(("down", tk[hi])); power += tk[hi]; score -= 1; hi -= 1
    else:
        break
assert log == [("up", 20), ("down", 130), ("up", 50), ("up", 80)] and best == 2 and score == 2
assert Solution().bagOfTokensScore(tk, 60) == oracle([20, 50, 80, 130], 60) == 2
print("Bài 2 OK:", log)

Bài 2 OK: [('up', 20), ('down', 130), ('up', 50), ('up', 80)]


### Bài 3

In [35]:
def two_pointers_answer(tokens, power):
    tokens = sorted(tokens)
    lo, hi = 0, len(tokens) - 1
    score = best = 0
    while lo <= hi:
        if power >= tokens[lo]:
            power -= tokens[lo]; score += 1; lo += 1; best = max(best, score)
        elif score > 0:
            power += tokens[hi]; score -= 1; hi -= 1
        else:
            break
    return best


assert [two_pointers_answer(list(t), p) for (t, p), _ in examples] == [0, 1, 2]
for t, p in tests[:400]:
    assert two_pointers_answer(list(t), p) == oracle(t, p)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def bagOfTokensScore(self, tokens, power):
        dq = deque(sorted(tokens))
        score = best = 0
        while dq:
            if power >= dq[0]:
                power -= dq.popleft()
                score += 1
                best = max(best, score)
            elif score > 0:
                power += dq.pop()
                score -= 1
            else:
                break
        return best


assert all(MySolutionAnswer().bagOfTokensScore(list(t), p) == w for (t, p), w in examples)
for t, p in tests[:400]:
    assert MySolutionAnswer().bagOfTokensScore(list(t), p) == oracle(t, p)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def maximize_greatness_answer(nums):
    a = sorted(nums)
    i = 0
    for x in a:                                          # x thuộc perm (đã sắp xếp)
        if x > a[i]:
            i += 1
    return i


def maximize_greatness_brute(nums):
    return max(sum(1 for p, x in zip(perm, nums) if p > x) for perm in set(itertools.permutations(nums)))


assert maximize_greatness_answer([1, 3, 5, 2, 1, 3, 1]) == 4 and maximize_greatness_answer([1, 2, 3, 4]) == 3
rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(0, 4) for _ in range(rng.randint(1, 7))]
    assert maximize_greatness_answer(a) == maximize_greatness_brute(a)
print("Bài 5 OK: khớp vét cạn hoán vị")

Bài 5 OK: khớp vét cạn hoán vị


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai thao tác đối nghịch: mua bằng phần tử rẻ nhất, bán phần tử đắt nhất):**

```python
class Solution:
    def bagOfTokensScore(self, tokens: List[int], power: int) -> int:
        tokens.sort()
        lo, hi = 0, len(tokens) - 1
        score = best = 0
        while lo <= hi:
            if power >= tokens[lo]:
                power -= tokens[lo]; score += 1; lo += 1
                best = max(best, score)
            elif score > 0:
                power += tokens[hi]; score -= 1; hi -= 1
            else:
                break
        return best
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp**; ngửa bằng token **rẻ nhất**, úp bằng token **đắt nhất**?
- [ ] Ưu tiên ngửa; chỉ úp khi không ngửa được và `score > 0`?
- [ ] Trả điểm **cực đại** (`best`), không phải điểm cuối?
- [ ] `while lo <= hi` (không bỏ sót token cuối cùng)?
- [ ] Xử lý `tokens` rỗng?
- [ ] Đã thử 3 ví dụ, token `0`, `power = 0`, token cuối cùng `lo == hi`?
- [ ] Đúng tên `class Solution`, `bagOfTokensScore(self, tokens, power)`, không in debug.